# Demo: das-to-store

This notebook demonstrates how to use the `das-to-store` package to convert Sintela ONYX  HDF5 files into Kerchunk reference JSON files, and then combine those JSON files into a single reference file for later access.

This demo is intended to run on WHOI's Hydra cluster, but the paths can be adapted to run on other systems.

## Setup

In [1]:
import fsspec
import time
from pathlib import Path

import xarray as xr
import zarr
from tqdm.notebook import tqdm

from das_to_store.kerchunk.generate import generate_json
from das_to_store.kerchunk.combine import combine_jsons
from das_to_store.interrogators.sintela import SintelaOnyxMultiZarrToZarrConfig

Define the path to the original DAS data (`.h5` files) and the path to the reference JSON files generated by kerchunk. Usually it's convienent to store reference files in a directory adjacent to the data (or in the same directory), but here they are saved to the current working directory for demonstration purposes.

In [2]:
# Path to data files.
MVCO_DAS_PATH = '/proj/das-onyx/mvco_2023-2024/'
SUBSET = 'decimator_mvco_20231218-20240209/'
DATA_DIR = Path(MVCO_DAS_PATH) / SUBSET

# Store reference files in the working directory.
REF_DIR = Path('./')
COMBINED_REF_FILENAME = DATA_DIR.name + '_combined.json'

Kerchunk uses the `fsspec` package for managing file access. Here we specify the remote file system (location of the data) and local file system (location for the generated references). In this case, both the remote and local file systems are the Hydra HPC and use a traditional `file` system (as opposed to an `S3` bucket, etc.).

In [3]:
fs_remote = fsspec.filesystem('file')  # file system to access the H5 files
fs_local = fsspec.filesystem('file')  # local file system to save final jsons to

List all of the `.h5` files to be processed by kerchunk.

In [4]:
flist = fs_remote.glob(str(DATA_DIR / '*.h5'))

len(flist)

77554

Here we'll process only the first 10 files for demonstration purposes.

In [5]:
flist = flist[:10]

flist

['/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_00.59.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.00.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.01.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.02.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.03.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.04.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.05.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.06.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209/decimator_2023-12-18_01.07.04_UTC.h5',
 '/proj/das-onyx/mvco_2023-2024/decimator_mvco_20231218-20240209

## Single file reference generation

The first step in using kerchunk is to generate a reference JSON file for each `.h5` file. The `generate_json` function creates the reference and writes it to the output directory at `fs_local` using the same file basename, but with a `.json` file extension in place of `.h5`.  This function is a wrapper around Kerchunk's `SingleHdf5ToZarr` class.

In [6]:
# Create the reference output directory if it doesn't exist.
REF_DIR.mkdir(exist_ok=True)

# Inline threshold adjusts the size (in bytes) below which binary blocks
# are included directly in the output. A higher threshold results in a
# larger JSON but faster loading time.

for f in tqdm(flist):
    generate_json(
        file_url=f,
        fs_remote=fs_remote,
        fs_local=fs_local,
        output_dir=REF_DIR,
        inline_threshold=300,
    )

  0%|          | 0/10 [00:00<?, ?it/s]

There should be as many JSON files in the output directory as there are `.h5` files in the input directory. Each JSON file contains the metadata and byte ranges needed to access the data in the corresponding `.h5` file.

In [7]:
json_list = fs_local.glob(str(REF_DIR / "*.json"))
json_list

['/user/jacob.davis/das-to-store/decimator_2023-12-18_00.59.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.00.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.01.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.02.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.03.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.04.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.05.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.06.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.07.04_UTC.json',
 '/user/jacob.davis/das-to-store/decimator_2023-12-18_01.08.04_UTC.json']

Individual reference files can be opened as if they were Zarr stores.

In [8]:
fo = str(REF_DIR / json_list[-1])

xr.open_dataset(
    "reference://",
    engine="zarr",
    group='Acquisition/Raw[0]',
    backend_kwargs={
        "consolidated": False,
        "storage_options": {
            "fo": fo,
            "remote_protocol": 'file',
            "target_protocol": 'file'
        }
    }
)

<xarray.Dataset> Size: 78MB
Dimensions:             (phony_dim_0: 15000, phony_dim_1: 1301)
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    RawData             (phony_dim_0, phony_dim_1) float32 78MB ...
    RawDataSampleCount  (phony_dim_0) float64 120kB ...
    RawDataTime         (phony_dim_0) float64 120kB ...
Attributes:
    NumberOfLoci:     1301
    OutputDataRate:   250.0
    RawDataUnit:      Radians
    StartLocusIndex:  0
    uuid:             ad903eed-fe01-4a7c-9b72-cd733dd95a72

## Combined reference generation

The next step is to combine the individual reference JSON files into a single reference JSON file that can be used to access all of the data.


In [9]:
COMBINED_REF_PATH = REF_DIR / COMBINED_REF_FILENAME

The `combine_jsons` function combines a list of JSON filepaths into a single, combined JSON file and saves it to `output_path`. This is a wrapper around the Kerchunk class `MultiZarrToZarr` which has several important keyword arguments that can be used to control how the JSON files are combined. Important arguments include `concat_dims`, which specifies the concatenatation dimensions, and `coo_map`, which enables extraction of coordinates from the individual JSON files and/or their metadata.

These keywords arguments are passed through the `combine_jsons` function via `mzz_config`. Several base configurations are defined in the `interrogators` module that can be used as starting points for different use cases.

Here we use the `SintelaOnyxMultiZarrToZarrConfig` defined in `interrogators/sintela.py` which is configured for Sintela Onyx DAS data. This configuration merges the attributes from the `'Acquisition'` and `'Acquisition/Raw[0]'` subgroups of the original `.h5` files, extracts a `time` coordinate from the `MeasurementStartTime` attribute, and concatenates files data along this new `'time'` dimension.

In [10]:
# Exclude the combined reference file from the list of JSON files to
# combine (in case it already exists).
json_list = [f for f in json_list if COMBINED_REF_FILENAME not in f]

combine_jsons(
    json_list=json_list,
    fs_local=fs_local,
    mzz_config=SintelaOnyxMultiZarrToZarrConfig(),
    remote_protocol='file',
    output_path=str(COMBINED_REF_PATH),
)

Now the combined files can be opened with xarray by treating the combined reference JSON file as a zarr store. The `reference://` protocol is used to indicate that the file is a Kerchunk reference file. Here we specify that the data is not consolidated (since it's a reference file).

Another benefit is that the combined files can be re-chunked. Here we demonstrate chunking along the `time` dimension for efficient access.

In [11]:
# Set "consolidated" to False since the metadata are stored across
# multiple JSON files.
backend_args = {
    "consolidated": False,
    "storage_options": {
        "fo": str(COMBINED_REF_PATH),
        "remote_protocol": 'file',
        "remote_options": {}
    }
}

# Open the dataset as a datatree since the data are nested within the
# Acquisition/Raw[0] group and attributes are in the root group.
das_dt = xr.open_datatree(
    "reference://",
    engine="zarr",
    backend_kwargs=backend_args,
    chunks={"time": 2, 'phony_dim_0': -1, 'phony_dim_1': -1},
)

# Merge root and Acquisition/Raw[0] groups.
das_ds = xr.merge([das_dt[''],  das_dt['Acquisition/Raw[0]']]).to_dataset()

das_ds

<xarray.Dataset> Size: 783MB
Dimensions:             (time: 10, phony_dim_0: 15000, phony_dim_1: 1301)
Coordinates:
  * time                (time) object 80B '2023-12-18T00:59:04.452000+00:00' ...
Dimensions without coordinates: phony_dim_0, phony_dim_1
Data variables:
    RawData             (time, phony_dim_0, phony_dim_1) float32 781MB dask.array<chunksize=(2, 15000, 1301), meta=np.ndarray>
    RawDataSampleCount  (time, phony_dim_0) float64 1MB dask.array<chunksize=(2, 15000), meta=np.ndarray>
    RawDataTime         (time, phony_dim_0) float64 1MB dask.array<chunksize=(2, 15000), meta=np.ndarray>
Attributes: (12/29)
    AcquisitionId:                /
    BandDataMaxUserValue:         0.0
    BandDataMinUserValue:         0.0
    Build:                        6.0.0-RC6_P
    CommitHash:                   0c754162
    DasInstrumentBox:             ONYXONYX 341
    ...                           ...
    StartLocusIndex:              0
    SystemType:                   Xavier
    TriggeredMeasurement:         0
    schemaVersion:                2.0
    OutputDataRate:               250.0
    RawDataUnit:                  Radians

## Profiling

Generation of the single and combined reference files happens once upfront. In the future, opening the dataset using the combined reference file should be much faster than opening the original `.h5` files directly.

In [12]:
start_time = time.perf_counter()

das_dt = xr.open_datatree(
    "reference://",
    engine="zarr",
    backend_kwargs=backend_args,
    chunks={"time": 2, 'phony_dim_0': -1, 'phony_dim_1': -1},
)

das_ds = xr.merge([das_dt[''],  das_dt['Acquisition/Raw[0]']]).to_dataset()

end_time = time.perf_counter()
elapsed_time_kerchunk = end_time - start_time
print(f"Elapsed time: {elapsed_time_kerchunk:.6f} seconds")


Elapsed time: 0.026529 seconds


In [13]:
start_time = time.perf_counter()

datasets = []
for file in flist:
    # Original H5 files contain global attributes in the Acquisition
    # group and data + attributes in the Acquisition/Raw[0] group.
    das_dt_h5 = xr.open_datatree(file, engine='h5netcdf', phony_dims="sort")
    das_ds_h5 = xr.merge([das_dt_h5['Acquisition'], das_dt_h5['Acquisition/Raw[0]']]).to_dataset()
    das_ds_h5['time'] = das_ds_h5.attrs.pop('MeasurementStartTime')

    # Only append datasets with full records.
    if das_ds_h5.sizes['phony_dim_0'] == 15000:
        datasets.append(das_ds_h5.set_coords('time'))

das_ds2 = xr.concat(datasets, dim='time')

end_time = time.perf_counter()
elapsed_time_h5 = end_time - start_time
print(f"Elapsed time: {elapsed_time_h5:.6f} seconds")

Elapsed time: 2.515943 seconds


In [14]:
print(f"Kerchunk open time: {elapsed_time_kerchunk:.6f} seconds")
print(f"HDF5 open time: {elapsed_time_h5:.6f} seconds")
print(f"Speedup: {elapsed_time_h5 / elapsed_time_kerchunk:.2f}x")

Kerchunk open time: 0.026529 seconds
HDF5 open time: 2.515943 seconds
Speedup: 94.84x
